# Avaliação Prática 2 — LSTM vs Transformer · executor Kaggle

Variante do [`colab.ipynb`](colab.ipynb) para rodar no Kaggle, cuja cota de GPU é
independente da do Colab — permitindo que esta avaliação treine **em paralelo** com a
Avaliação Prática 1 rodando no Colab.

Como no Colab: o notebook é o ambiente de execução, os scripts versionados são o artefato.

---

## Antes de rodar — dois interruptores no painel direito (`⋮` → Settings)

**1. `Accelerator` → `GPU T4 x2` (ou `P100`).**
Sem isso o BERT roda em CPU e leva horas.

**2. `Internet` → `On`.**
Este é o que falha em silêncio: sem internet, o `git clone` e o download do BERTimbau
quebram com erros de rede que não dizem "ligue a internet". O Kaggle exige conta
verificada por telefone para habilitar a opção.

## E os dados

O corpus é material da disciplina e **não está no repositório**. No painel direito:
`+ Add Input` → `Upload` → `New Dataset` → envie `g1_v1_ws.csv` e `g1_v2_ws.csv` →
dê um título (ex.: `g1-emocoes`). Ele aparecerá em `/kaggle/input/g1-emocoes/`.

A célula 2 localiza os CSVs em qualquer subpasta de `/kaggle/input`, então o nome que
você escolher não importa.

In [ ]:
# 1. Clonar o repositório e instalar as dependências.
#    O Kaggle já traz TensorFlow, PyTorch, scikit-learn e NLTK; na prática só
#    `transformers` costuma faltar.
import pathlib, subprocess, sys

REPO = pathlib.Path('/kaggle/working/machine-learning-fundamentals')
ACTIVITY = REPO / 'activities' / 'avaliacao-pratica-2'

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/fsd-dantas/machine-learning-fundamentals.git',
                    str(REPO)], check=True)

assert ACTIVITY.is_dir(), f'{ACTIVITY} não existe no repositório publicado.'
%cd {ACTIVITY}
!pip install -q -r requirements.txt

import torch
print('PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Sem GPU: Settings → Accelerator → GPU T4 x2'

In [ ]:
# 2. Trazer o corpus do dataset de entrada para data/.
#    Busca recursiva em /kaggle/input: funciona qualquer que seja o nome que você deu
#    ao dataset, e falha com mensagem clara se você esqueceu de anexá-lo.
import shutil, pathlib

DATA = pathlib.Path('data'); DATA.mkdir(exist_ok=True)
WANTED = ('g1_v1_ws.csv', 'g1_v2_ws.csv')

found = {}
for path in pathlib.Path('/kaggle/input').rglob('*.csv'):
    if path.name in WANTED:
        found[path.name] = path

missing = [n for n in WANTED if n not in found]
assert not missing, (
    f'não encontrei {missing} em /kaggle/input. Anexe os CSVs: painel direito → '
    '+ Add Input → Upload → New Dataset.')

for name, path in found.items():
    shutil.copy(path, DATA / name)
    print(f'{name}  ({path.stat().st_size/1024:.0f} KB)  <- {path.parent}')

In [ ]:
# 3. Conferir o corpus ANTES de gastar GPU.
#    Devem aparecer 292 duplicatas e 4 conflitos de rótulo, restando 2.436 textos.
#    Número diferente = problema no upload, e é melhor saber agora.
import common

for task in ('binary', 'multiclass'):
    common.load_corpus(task)
    print()

In [ ]:
# 4. ETAPA CORE — 2 tarefas × {majoritária, TF-IDF+SVM, LSTM, BERT} (~8 min).
#    Ao fim desta célula a entrega já é completa: acurácia, F1 e matriz de confusão
#    nas duas tarefas, com LSTM e Transformer comparados por McNemar pareado.
!python run_all.py --stage core
!python report.py

In [ ]:
# 5. ETAPA SEEDS — sementes 7 e 2024 (~15 min).
#    O teste tem ~730 textos: erro padrão binomial de ~1,5 pp. Uma única partição
#    ordena dois modelos por sorte — o baseline sozinho varia 51,2% → 55,3% na
#    multiclasse só trocando a semente.
!python run_all.py --stage seeds
!python report.py

In [ ]:
# 6. SENSIBILIDADE + EXTRAS (~6 min).
#    sensitivity: tarefa binária SEM `neutro`/`surpresa` — a conclusão depende da
#                 decisão do professor de chamá-los de 'positivo'?
#    extras:      BiLSTM — a bidirecionalidade resgata o braço recorrente?
!python run_all.py --stage sensitivity --stage extras
!python report.py

In [ ]:
# 7. Empacotar os resultados em /kaggle/working (aba Output → Download).
#    Só JSONs e PNGs: são a evidência e permitem regenerar o relatório sem GPU.
import shutil, pathlib

OUT = pathlib.Path('/kaggle/working/ap2-results')
if OUT.exists():
    shutil.rmtree(OUT)
shutil.copytree('results', OUT / 'results')

img_src = pathlib.Path('../../assets/img')
(OUT / 'img').mkdir()
for png in img_src.glob('avaliacao-pratica-2-*.png'):
    shutil.copy(png, OUT / 'img' / png.name)

shutil.make_archive('/kaggle/working/ap2-results', 'zip', OUT)
print('pronto: /kaggle/working/ap2-results.zip')
print(f"{len(list((OUT / 'results').glob('*.json')))} artefatos de resultado")